# Generate an AI red-team report from Garak output

This notebook renders a responsive HTML report from Garak `.report.jsonl` files, optionally with hit-log evidence. It supports a single scan or a baseline/defended comparison. Edit the paths and metadata below. The first scan is the baseline unless `metadata['baseline']` names another label.

Garak cautions that scores are not normalized across probes. The report therefore labels scan-wide rates as weighted detector-evaluation ASR and calculates a separate comparison on matched probe/detector pairs. Review scanner findings and redact sensitive hit-log content before sharing.

In [ ]:
from pathlib import Path
import sys

candidate_roots = [Path.cwd(), Path.cwd().parent]
demo_root = next(
    (root.resolve() for root in candidate_roots if (root / 'scripts/garak_report_generator.py').exists()),
    None,
)
if demo_root is None:
    raise FileNotFoundError('Start Jupyter from evalhub-demo/ or evalhub-demo/notebooks/.')
script_path = demo_root / 'scripts/garak_report_generator.py'
sys.path.insert(0, str(script_path.resolve().parent))
import garak_report_generator as reportgen
print('Using report generator:', script_path.resolve())

## Inputs and report context

Download the native Garak JSONL artifacts into `evalhub-demo/runs/` before running these cells. Architecture, known issues, and recommendations are human-authored context: Garak's score files do not contain enough evidence to infer those reliably. `hitlog_files` can be left empty.

In [ ]:
run_dir = demo_root / 'runs'
scan_files = {
    'Baseline': run_dir / 'Baseline/scan.report.jsonl',
    'Guardrailed': run_dir / 'Guardrailed/scan.report.jsonl',
}

# Optional: include these only when it is appropriate to put prompts and outputs in the report.
hitlog_files = {
    # 'Guardrailed': run_dir / 'Guardrailed/scan.hitlog.jsonl',
}
include_examples = False  # Set True only after reviewing hit-log sensitivity.
output_path = demo_root / 'reports/garak-red-team-report.html'

metadata = {
    'title': 'AI Red Teaming Report',
    'subtitle': 'Garak vulnerability scan and defense evaluation',
    'benchmark': 'OWASP LLM Top 10',
    'platform': 'Describe the serving and deployment platform',
    'generated': None,  # Optional ISO date; None uses today's date.
    'taxonomy': 'owasp',
    'baseline': 'Baseline',
    'architecture': 'Describe request flow and the locations of the defenses under evaluation.',
    'category_labels': {
        'llm01': 'LLM01 — Prompt Injection',
        'llm02': 'LLM02 — Sensitive Information Disclosure',
        'llm03': 'LLM03 — Supply Chain',
        'llm04': 'LLM04 — Data and Model Poisoning',
        'llm05': 'LLM05 — Improper Output Handling',
        'llm06': 'LLM06 — Excessive Agency',
        'llm07': 'LLM07 — System Prompt Leakage',
        'llm08': 'LLM08 — Vector and Embedding Weaknesses',
        'llm09': 'LLM09 — Misinformation',
        'llm10': 'LLM10 — Unbounded Consumption',
    },
    'scans': {
        'Baseline': {
            'description': 'Describe the model endpoint and configuration.',
            'layers': ['Garak probes', 'Model endpoint'],
        },
        'Guardrailed': {
            'description': 'Describe the model, guardrails, and configuration.',
            'layers': ['Garak probes', 'Guardrails and classifier', 'Model endpoint'],
        },
    },
    'methodology': [
        'State probe selection, detector configuration, generation count, and run conditions.',
        'ASR is calculated as (total_evaluated - passed) / total_evaluated.',
        'Run-wide rates are weighted descriptive summaries; they are not normalized across probes.',
    ],
    'known_issues': [],  # Add detector artifacts, scanner bugs, exclusions, and workarounds.
    'defense_analysis': [],  # Add human-reviewed layer-specific findings tied to probe evidence.
    'recommendations': [],  # Add evidence-linked actions and priority; otherwise the report lists triage items.
    'max_examples': 6,
}

# Remove an optional date override when it is unset.
if metadata['generated'] is None:
    metadata.pop('generated')

## Load, compare, and render

A matched comparison uses only identical probe/detector pairs. The full tables retain run-specific coverage and show missing pairs as `Not evaluated`.

In [ ]:
missing = [str(path) for path in scan_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Update scan_files; missing: ' + ', '.join(missing))

scans = {
    label: reportgen.load_garak_report(path, taxonomy=metadata.get('taxonomy'))
    for label, path in scan_files.items()
}
hitlogs = {
    label: reportgen.load_hitlog(path)
    for label, path in hitlog_files.items()
}

output_path.parent.mkdir(parents=True, exist_ok=True)
reportgen.write_report(
    scans,
    output_path=output_path,
    metadata=metadata,
    hitlogs=hitlogs,
    include_examples=include_examples,
)
print('Report written to:', output_path.resolve())

## Review before distribution

Check that configurations had comparable probe and detector coverage; confirm whether hit examples are safe to share; review detector artifacts and known tool issues; and ensure each recommendation is supported by the recorded evidence. The report is an evaluation artifact, not a standalone certification of system safety.